#### Modules

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report

#### Train and testing splits 

Stratify ensures training, validation and testing datasets have the exact same proportion of crack and uncrack as the original dataset

In [13]:
# Load datasets
df_walls = pd.read_pickle('wall_data.pkl')
df_decks = pd.read_pickle('deck_data.pkl')

# Split the Walls DataFrame (15% of total wall data for validation, hence 0.15 / 0.85 ≈ 0.176 for second split)
walls, walls_te = train_test_split(df_walls, test_size=0.2, random_state=42, stratify=df_walls['Label'])
walls_tr, walls_val = train_test_split(walls, test_size=0.176, random_state=42, stratify=walls['Label'])

# Split the Decks DataFrame
decks, decks_te = train_test_split(df_decks, test_size=0.2, random_state=42, stratify=df_decks['Label'])
decks_tr, decks_val = train_test_split(decks, test_size=0.176, random_state=42, stratify=decks['Label'])

#### CNN Architecture

In [15]:
def cnn(input_shape):
    
    model = models.Sequential()

    # Filter 1
    model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(layers.MaxPooling2D((2, 2)))

    # Filter 2
    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Filter 3
    model.add(layers.Conv2D(128, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Set up input for neural network for classification
    model.add(layers.Flatten())

    # Layer 1 (Dropout to prevent overfitting)
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.5))

    # Layer 2 (Dropout to prevent overfitting)
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(0.5))
    
    # Output layer - use sigmoid to output a probability between 0.0 (Non-Cracked) and 1.0 (Cracked)
    model.add(layers.Dense(1, activation='sigmoid'))

    # Adam is a smart optimizer, Binary Crossentropy is standard for 2 classes
    model.compile(optimizer='adam', loss='binary_crossentropy',metrics=['accuracy'])
    
    return model

#### Training phase

In [ ]:
x_tr_walls = np.stack(walls_tr['ImageData'].values)
x_val_walls = np.stack(walls_val['ImageData'].values)
x_te_walls = np.stack(walls_te['ImageData'].values)

# 2. Extract the binary labels (0 for Uncracked, 1 for Cracked)
y_tr_walls = walls_tr['LabelCode'].values
y_val_walls = walls_val['LabelCode'].values
y_te_walls = walls_te['LabelCode'].values

# 3. Normalize the pixel values! (Crucial for CNNs)
x_tr_walls = x_tr_walls / 255.0
x_val_walls = x_val_walls / 255.0
x_te_walls = x_te_walls / 255.0

(3,)